<a href="https://colab.research.google.com/github/HarisBinHabib/Internship-Repo/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
if not os.path.exists("/content/repo"):
    !git clone https://github.com/HarisBinHabib/Internship-Repo.git /content/repo
%cd /content/repo
!pip install duckdb scikit-learn --quiet

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

os.makedirs("work/outputs", exist_ok=True)
print("Ready.")

/content/repo
Ready.


# Capstone — Refresh / Content Opportunity Scoring

**Question:** Which pages should an editor review first for refresh, given limited review
capacity, using signals knowable before the decision point?

**Decision:** editor prioritization of a fixed weekly review budget.
**Label:** a real future-observed outcome — decline in impressions over the 30 days
**after** the decision point — not a same-window proxy like the starter's `trend_direction`.
**Validation:** client-grouped train/test split, so no client appears in both sets.
**Metric:** precision@50, compared against the transparent rule baseline from w04.

In [11]:
frame = con.sql(f"""
    WITH anchor AS (SELECT DATE '2026-03-31' AS d),
    feat AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(f.gsc_impressions) AS imp_prev90,
            SUM(f.gsc_clicks)      AS clk_prev90,
            AVG(f.gsc_avg_position) AS pos_prev90,
            MAX(f.report_date)     AS last_seen_prev90
        FROM {TABLES['fact_daily']} f, anchor a
        WHERE f.report_date > a.d - INTERVAL 90 DAY AND f.report_date <= a.d
        GROUP BY 1, 2
        HAVING imp_prev90 >= 100
    ),
    label AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(f.gsc_impressions) AS imp_next30
        FROM {TABLES['fact_daily']} f, anchor a
        WHERE f.report_date > a.d AND f.report_date <= a.d + INTERVAL 30 DAY
        GROUP BY 1, 2
    )
    SELECT
        feat.*, COALESCE(label.imp_next30, 0) AS imp_next30
    FROM feat
    LEFT JOIN label USING (client_hash_id, content_hash_id)
""").df()

import pandas as pd

frame['ctr_prev90'] = frame['clk_prev90'] / frame['imp_prev90']
frame['days_since_seen'] = (
    pd.Timestamp('2026-03-31') - pd.to_datetime(frame['last_seen_prev90'])
).dt.days
frame['declined_next30'] = (frame['imp_next30'] < 0.70 * frame['imp_prev90']).astype(int)

print(f"{len(frame):,} (client, content) pairs")
print(f"Base rate (declined_next30): {frame['declined_next30'].mean():.3f}")
frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,258 (client, content) pairs
Base rate (declined_next30): 0.775


,client_hash_id,content_hash_id,imp_prev90,clk_prev90,pos_prev90,last_seen_prev90,imp_next30,ctr_prev90,days_since_seen,declined_next30
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,654.0,1.0,31.202735,2026-03-31,187.0,0.001529,0,1
1,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,1726.0,2.0,8.799970,2026-03-31,202.0,0.001159,0,1
2,client_62f4a7e64f5e0096,content_3f87f49c36774e23,290.0,0.0,27.505333,2026-03-31,93.0,0.000000,0,1
3,client_62f4a7e64f5e0096,content_57c94ea6f9e9b1b2,112.0,0.0,6.529398,2026-03-31,31.0,0.000000,0,1
4,client_62f4a7e64f5e0096,content_64706c8afebebb8c,1322.0,2.0,6.527702,2026-03-31,131.0,0.001513,0,1


In [12]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['imp_prev90', 'clk_prev90', 'pos_prev90', 'ctr_prev90', 'days_since_seen']
X = frame[feature_cols].fillna(0)
y = frame['declined_next30']
groups = frame['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Clients overlapping train/test (must be 0): {len(overlap)}")
print(f"Train rows: {len(X_train):,}   Test rows (before activity filter): {len(test_idx):,}")

Clients overlapping train/test (must be 0): 0
Train rows: 108,675   Test rows (before activity filter): 11,583


In [13]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

test_frame_full = frame.iloc[test_idx].copy()

# Exclude pages that are already fully dormant (no activity in the last 30 days of the
# feature window) — predicting "stays at zero" for an already-dead page isn't a meaningful
# decline prediction, and inflates precision artificially. We evaluate only on pages that
# still had some recent activity, where the outcome is genuinely uncertain.
still_active = test_frame_full['days_since_seen'] < 30
test_frame = test_frame_full[still_active].copy()

test_frame['baseline_score'] = (
    (test_frame['imp_prev90'] >= 500).astype(int) * test_frame['imp_prev90']
)

X_test = test_frame[feature_cols].fillna(0)
y_test = test_frame['declined_next30']

k = 50
baseline_p_at_k = precision_at_k(test_frame['baseline_score'], y_test.values, k)
base_rate = y_test.mean()
print(f"Active pages only: {len(test_frame):,} of {len(test_frame_full):,} test rows")
print(f"Baseline precision@{k}: {baseline_p_at_k:.3f}  (base rate: {base_rate:.3f})")

Active pages only: 6,858 of 11,583 test rows
Baseline precision@50: 0.640  (base rate: 0.533)


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
}

results = {"baseline": {"precision_at_50": round(baseline_p_at_k, 3), "roc_auc": None}}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    p_at_k = precision_at_k(proba, y_test.values, k)
    auc = roc_auc_score(y_test, proba)
    results[name] = {"precision_at_50": round(p_at_k, 3), "roc_auc": round(auc, 3)}
    print(f"{name:22} precision@{k} = {p_at_k:.3f}   ROC-AUC = {auc:.3f}")

print(f"\nBase rate: {base_rate:.3f}")

import json
with open("work/outputs/capstone_model_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: work/outputs/capstone_model_results.json")

logistic_regression    precision@50 = 0.680   ROC-AUC = 0.581
random_forest          precision@50 = 0.560   ROC-AUC = 0.615

Base rate: 0.533

Saved: work/outputs/capstone_model_results.json


In [15]:
leakage_checklist = {
    "features_calculated_after_decision_point": False,
    "feature_window_overlaps_target_window": False,
    "product_flags_used_as_features": False,
    "derived_field_encodes_target": False,
    "duplicate_rows_split_across_train_test": len(overlap) > 0,
}
for k_, v in leakage_checklist.items():
    print(f"{k_}: {'FAIL' if v else 'pass'}")

features_calculated_after_decision_point: pass
feature_window_overlaps_target_window: pass
product_flags_used_as_features: pass
derived_field_encodes_target: pass
duplicate_rows_split_across_train_test: pass


In [16]:
best_model_name = max(
    (n for n in results if n != "baseline"),
    key=lambda n: results[n]["precision_at_50"]
)
best_model = models[best_model_name]

test_frame['model_probability'] = best_model.predict_proba(X_test)[:, 1]
test_frame['reason_code'] = np.where(
    test_frame['baseline_score'] > 0, 'stale_visible_page', 'model_decline_risk'
)
test_frame['action'] = 'refresh'

ranked = test_frame.sort_values('model_probability', ascending=False).reset_index(drop=True)
output_cols = ['client_hash_id', 'content_hash_id', 'model_probability', 'reason_code',
               'action', 'imp_prev90', 'ctr_prev90', 'pos_prev90', 'declined_next30']
ranked[output_cols].to_csv("work/outputs/capstone_ranked_queue.csv", index=False)

print(f"Best model: {best_model_name}")
print(f"Ranked queue written: {len(ranked)} rows")
ranked[output_cols].head(20)

Best model: logistic_regression
Ranked queue written: 6858 rows


,client_hash_id,content_hash_id,model_probability,reason_code,action,imp_prev90,ctr_prev90,pos_prev90,declined_next30
0,client_20259bd6705d81d4,content_82e35c4845e6c391,0.995843,stale_visible_page,refresh,177698.0,0.000439,18.833815,1
1,client_65de48885f4ef01b,content_62673eea26c31c17,0.987117,stale_visible_page,refresh,146717.0,0.001077,6.585616,1
2,client_20259bd6705d81d4,content_0adb360f9005b515,0.954054,stale_visible_page,refresh,102053.0,0.000950,9.189993,1
3,client_20259bd6705d81d4,content_a2a286258ce3f1f8,0.938248,stale_visible_page,refresh,88945.0,0.000753,3.601463,1
4,client_20259bd6705d81d4,content_761ad39548ba1d60,0.909962,stale_visible_page,refresh,83928.0,0.001406,21.486852,0
5,client_20259bd6705d81d4,content_8c4e4df7a2d9f010,0.904314,stale_visible_page,refresh,93230.0,0.003132,4.574189,1
6,client_20259bd6705d81d4,content_3c20261c0a7c01be,0.899587,stale_visible_page,refresh,90061.0,0.002998,5.961711,1
7,client_20259bd6705d81d4,content_3f8597ccc4b874a9,0.896197,stale_visible_page,refresh,113242.0,0.005042,5.941204,0
8,client_20259bd6705d81d4,content_89c10d52fc81ac39,0.894781,stale_visible_page,refresh,93337.0,0.003182,20.813117,1
9,client_20259bd6705d81d4,content_0b7e2cd65fadec6f,0.881694,stale_visible_page,refresh,75831.0,0.001503,30.074190,0


## Limitations

- This result comes from a single mid-panel decision point (2026-03-31) on the warehouse
  release — not yet validated across multiple time windows.
- Client history depth is uneven (per `dim_clients.gsc_data_start`); some clients contribute
  thin feature windows.
- The label (`declined_next30`) is a threshold-based proxy for "decline" — a real editorial
  outcome (traffic recovery after refresh) would require an actual experiment.
- Results are observational, not causal — a refresh recommendation is decision-support, not a
  guarantee.